# Alerta Inteligente: ajuste fino do BERTimbau para classificar relatos

Grupo Sentinela do Cachoeira. Tarefa: classificar os relatos que os moradores mandam ao bot em
**confirmado**, **observacao** ou **nao_relacionado**.

- Dados: 300 relatos fictícios (`dados/finetuning/relatos.csv`), divididos por frase base em treino, validação e teste.
- Modelo base: `neuralmind/bert-base-portuguese-cased` (BERTimbau).
- Métrica principal: F1-macro no conjunto de teste.
- Comparações: o mesmo BERTimbau antes do ajuste, TF-IDF + regressão logística e o Gemini sem treino.

Antes de rodar: **Ambiente de execução > Alterar o tipo de ambiente de execução > GPU T4**.
Para a comparação com o Gemini, guarde a chave em **Secrets** (ícone de chave na barra lateral) com o nome `GEMINI_API_KEY`.

In [ ]:
# 1. Baixa o repositório do projeto (público) e confere as bibliotecas
!git clone -q https://github.com/DiegoZarpok/alerta-inteligente.git
import torch, transformers, sklearn, pandas as pd
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (vai ficar lento)")

In [ ]:
# 2. Dados
import random, numpy as np
SEMENTE = 42
random.seed(SEMENTE); np.random.seed(SEMENTE); torch.manual_seed(SEMENTE)

dados = pd.read_csv("alerta-inteligente/dados/finetuning/relatos.csv")
CLASSES = ["confirmado", "observacao", "nao_relacionado"]
rotulo = {c: i for i, c in enumerate(CLASSES)}
treino = dados[dados.divisao == "treino"].reset_index(drop=True)
validacao = dados[dados.divisao == "validacao"].reset_index(drop=True)
teste = dados[dados.divisao == "teste"].reset_index(drop=True)
print(pd.crosstab(dados.classe, dados.divisao))

In [ ]:
# 3. Tokenizador e modelo base
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset

BASE = "neuralmind/bert-base-portuguese-cased"
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
tokenizador = AutoTokenizer.from_pretrained(BASE)

def carregar(df, embaralhar):
    enc = tokenizador(list(df.texto), truncation=True, padding="max_length", max_length=96, return_tensors="pt")
    y = torch.tensor([rotulo[c] for c in df.classe])
    return DataLoader(TensorDataset(enc["input_ids"], enc["attention_mask"], y), batch_size=16, shuffle=embaralhar)

dl_treino, dl_val, dl_teste = carregar(treino, True), carregar(validacao, False), carregar(teste, False)

def novo_modelo():
    return AutoModelForSequenceClassification.from_pretrained(
        BASE, num_labels=3, id2label=dict(enumerate(CLASSES)), label2id=rotulo).to(dispositivo)

In [ ]:
# 4. Funções de avaliação
import time
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

@torch.no_grad()
def prever(modelo, dl):
    modelo.eval()
    saida = []
    for ids, mascara, _ in dl:
        logits = modelo(input_ids=ids.to(dispositivo), attention_mask=mascara.to(dispositivo)).logits
        saida += logits.argmax(-1).cpu().tolist()
    return [CLASSES[i] for i in saida]

def medir(nome, verdade, previsto):
    r = {"modelo": nome, "f1_macro": round(f1_score(verdade, previsto, average="macro"), 3),
         "acuracia": round(accuracy_score(verdade, previsto), 3)}
    print(r)
    return r

resultados = []

In [ ]:
# 5. Medição do modelo base ANTES do ajuste (camada de classificação ainda sem treino)
torch.manual_seed(SEMENTE)
base = novo_modelo()
resultados.append(medir("BERTimbau base, sem ajuste", list(teste.classe), prever(base, dl_teste)))
del base

In [ ]:
# 6. Ajuste fino
torch.manual_seed(SEMENTE)
modelo = novo_modelo()
EPOCAS, TAXA = 6, 3e-5
otimizador = torch.optim.AdamW(modelo.parameters(), lr=TAXA, weight_decay=0.01)
passos = EPOCAS * len(dl_treino)
agenda = get_linear_schedule_with_warmup(otimizador, int(0.1 * passos), passos)

melhor_f1, melhor_estado, historico = -1, None, []
inicio = time.time()
for epoca in range(1, EPOCAS + 1):
    modelo.train()
    perda_total = 0
    for ids, mascara, y in dl_treino:
        otimizador.zero_grad()
        saida = modelo(input_ids=ids.to(dispositivo), attention_mask=mascara.to(dispositivo), labels=y.to(dispositivo))
        saida.loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        otimizador.step(); agenda.step()
        perda_total += saida.loss.item()
    f1_val = f1_score(list(validacao.classe), prever(modelo, dl_val), average="macro")
    historico.append({"epoca": epoca, "perda_treino": round(perda_total / len(dl_treino), 4), "f1_validacao": round(f1_val, 3)})
    print(historico[-1])
    if f1_val > melhor_f1:
        melhor_f1 = f1_val
        melhor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
tempo_treino = round(time.time() - inicio, 1)
modelo.load_state_dict(melhor_estado)
print(f"Treino em {tempo_treino} s; melhor F1 na validação: {melhor_f1:.3f}")

In [ ]:
# 7. Medição no TESTE (frases que o modelo nunca viu)
previsto_bert = prever(modelo, dl_teste)
r_bert = medir("BERTimbau ajustado", list(teste.classe), previsto_bert)
inicio = time.perf_counter(); prever(modelo, dl_teste); r_bert["latencia_ms_por_relato"] = round((time.perf_counter() - inicio) / len(teste) * 1000, 2)
resultados.append(r_bert)
print(classification_report(teste.classe, previsto_bert, labels=CLASSES, digits=3))
print(pd.DataFrame(confusion_matrix(teste.classe, previsto_bert, labels=CLASSES), index=CLASSES, columns=CLASSES))

In [ ]:
# 8. Comparação: TF-IDF + regressão logística
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=2000, class_weight="balanced"))
tfidf.fit(treino.texto, treino.classe)
resultados.append(medir("TF-IDF + regressão logística", list(teste.classe), list(tfidf.predict(teste.texto))))

In [ ]:
# 9. Comparação: Gemini sem treino (mesmo conjunto de teste, prompt da pasta prompts/)
import json, re, requests
from google.colab import userdata
MODELO_GEMINI = "gemini-3.5-flash-lite"
texto_prompt = open("alerta-inteligente/prompts/classificar_relato_v1.md", encoding="utf-8").read()
sistema = re.search(r"## Sistema\n(.*?)\n## Usuario", texto_prompt, re.S).group(1).strip()
modelo_usuario = re.search(r"## Usuario\n(.*)$", texto_prompt, re.S).group(1).strip()
try:
    chave = userdata.get("GEMINI_API_KEY")
except Exception:
    chave = None
if not chave:
    print("Sem GEMINI_API_KEY nos Secrets: pulando esta comparação.")
else:
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODELO_GEMINI}:generateContent"
    previsto_g, tempos, tokens, falhas = [], [], 0, 0
    for texto in teste.texto:
        corpo = {"systemInstruction": {"parts": [{"text": sistema}]},
                 "contents": [{"role": "user", "parts": [{"text": modelo_usuario.replace("{texto}", texto)}]}],
                 "generationConfig": {"responseMimeType": "application/json", "maxOutputTokens": 30,
                                      "thinkingConfig": {"thinkingLevel": "minimal"}}}
        for tentativa in range(4):
            t = time.perf_counter()
            r = requests.post(url, headers={"x-goog-api-key": chave}, json=corpo, timeout=60)
            if r.status_code == 200: break
            time.sleep(5 * (tentativa + 1))
        tempos.append(time.perf_counter() - t)
        try:
            resposta = r.json()
            tokens += resposta.get("usageMetadata", {}).get("totalTokenCount", 0)
            classe = json.loads(resposta["candidates"][0]["content"]["parts"][0]["text"]).get("classe")
        except Exception:
            classe = None
        if classe not in CLASSES:
            falhas += 1  # sem resposta válida (ex.: limite 429): conta como erro, não some da conta
            classe = "nao_relacionado"
        previsto_g.append(classe if classe in CLASSES else "nao_relacionado")
        time.sleep(4)  # limite do nível gratuito
    r_g = medir(f"Gemini sem treino ({MODELO_GEMINI})", list(teste.classe), previsto_g)
    r_g["latencia_ms_por_relato"] = round(np.median(tempos) * 1000, 1)
    r_g["tokens_total"] = tokens
    r_g["respostas_invalidas_ou_sem_resposta"] = falhas
    print(f"Gemini: {falhas} de {len(teste)} relatos sem resposta válida (contados como erro)")
    resultados.append(r_g)

In [ ]:
# 10. Tabela final e arquivos para o relatório
tabela = pd.DataFrame(resultados)
print(tabela.to_string(index=False))
saida = {"base": BASE, "epocas": EPOCAS, "taxa_aprendizado": TAXA, "tempo_treino_s": tempo_treino,
         "historico": historico, "tamanhos": {"treino": len(treino), "validacao": len(validacao), "teste": len(teste)},
         "resultados": resultados,
         "matriz_confusao_bertimbau": confusion_matrix(teste.classe, previsto_bert, labels=CLASSES).tolist(),
         "classes": CLASSES}
json.dump(saida, open("finetuning_resultados.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

modelo.save_pretrained("bertimbau_relatos"); tokenizador.save_pretrained("bertimbau_relatos")
from google.colab import files
files.download("finetuning_resultados.json")
# O modelo ajustado tem cerca de 400 MB. Não é obrigatório para a entrega (o sistema local usa o
# classificador TF-IDF); para baixar, mude BAIXAR_MODELO para True e rode esta célula de novo.
BAIXAR_MODELO = False
if BAIXAR_MODELO:
    !zip -qr bertimbau_relatos.zip bertimbau_relatos
    files.download("bertimbau_relatos.zip")